In [1]:
from fastapi import APIRouter, Query, HTTPException
from typing import Optional, List, Dict, Any, Literal
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import sys

sys.path.insert(0, r"D:/A0_Project")  # 讓它優先被找到
from PI_SYSTEM.models.sql_db_connect import MySQLConnet

db = MySQLConnet('piaoi_ol_defect_map')
cim_db = MySQLConnet('cim_piaoi')
old_db = MySQLConnet('l6a01_project')

#db = MySQLConnet('piaoi_inspection_density')
#from PI_SYSTEM.models.density.cim_density_job import Config as DensityJobConfig
#from PI_SYSTEM.models.density.API_Config import API_Config

In [4]:
db = MySQLConnet('piaoi_capa')
all_tbns = db.list_tables()

#for tbn in [v for v in all_tbns if '202605' in v]:
    #db.drop_table(tbn)

2026-05-07 13:21:25,730 - INFO - [list_tables] 成功取得資料表名稱，共 18 張表。
2026-05-07 13:21:26,930 - INFO - [drop_table] 資料表 'aoi100_202605_capa_glass_table' 已刪除.
2026-05-07 13:21:27,663 - INFO - [drop_table] 資料表 'aoi100_202605_capa_hourly_rawdata' 已刪除.
2026-05-07 13:21:28,021 - INFO - [drop_table] 資料表 'aoi100_202605_capa_summary' 已刪除.
2026-05-07 13:21:28,564 - INFO - [drop_table] 資料表 'aoi200_202605_capa_glass_table' 已刪除.
2026-05-07 13:21:28,943 - INFO - [drop_table] 資料表 'aoi200_202605_capa_hourly_rawdata' 已刪除.
2026-05-07 13:21:29,366 - INFO - [drop_table] 資料表 'aoi200_202605_capa_summary' 已刪除.
2026-05-07 13:21:29,897 - INFO - [drop_table] 資料表 'aoi300_202605_capa_glass_table' 已刪除.
2026-05-07 13:21:30,406 - INFO - [drop_table] 資料表 'aoi300_202605_capa_hourly_rawdata' 已刪除.
2026-05-07 13:21:30,787 - INFO - [drop_table] 資料表 'aoi300_202605_capa_summary' 已刪除.


In [13]:
all_tbns = cim_db.list_tables()
pi_tbns = [v for v in all_tbns if 'cim_pi_glass' in v ]
raw_tbns = [v for v in all_tbns if 'cim_defect']



2026-05-07 13:55:00,244 - INFO - [list_tables] 成功取得資料表名稱，共 144 張表。


In [16]:
df = cim_db.get_table('cim_pi_glass_202600')
print(len(df))
df.iloc[-1,:].to_dict()
df['pi_time'].unique()

3916


<DatetimeArray>
['NaT']
Length: 1, dtype: datetime64[ns]

In [ ]:
for tbn in ['cim_pi_glass_202600']:#pi_tbns:
    rows = cim_db.get_rows(tbn, {'aoi': 'CAAOI300'})
    if len(rows) ==0:
        continue
    df = pd.DataFrame(rows)
    for (aoi, pi_type), raws in df.groupby(['aoi', 'pi_type']):
        print(aoi, pi_type, len(raws))
        print(raws[['line_id',  'pi_type', 'pi_time', 'test_time', 'recipe_id']].head())

CAAOI300 API 229
    line_id pi_type             pi_time           test_time recipe_id
0  CAPIC700     API 2026-05-01 04:22:12 2026-05-01 05:38:55     03024
1  CAPIC700     API 2026-05-01 04:22:12 2026-05-01 05:44:19     03024
2  CAPIC700     API 2026-05-01 04:22:12 2026-05-01 05:49:45     03024
3  CAPIC700     API 2026-05-01 04:22:12 2026-05-01 05:55:16     03024
4  CAPIC700     API 2026-05-01 04:22:12 2026-05-01 06:00:26     03024
CAAOI300 BPI 28
      line_id pi_type             pi_time           test_time recipe_id
170  CAPIC700     BPI 2026-05-03 23:55:27 2026-05-03 17:38:53     01024
171  CAPIC700     BPI 2026-05-03 23:55:27 2026-05-03 17:44:49     01024
172  CAPIC700     BPI 2026-05-03 23:55:27 2026-05-03 17:33:42     01024
206  CAPIC600     BPI 2026-05-05 21:51:09 2026-05-05 14:13:51     01145
207  CAPIC600     BPI 2026-05-05 21:51:09 2026-05-05 14:19:05     01145


In [37]:
from API_Config import API_Config
def _recipe_family(recipe_id: object) -> str:
    """
    依 recipe_id 粗分頁籤用途。
    """
    s = "" if recipe_id is None else str(recipe_id).strip()

    if len(s) == 4:
        if s.startswith("2"):
            return "UPI"
        if s.startswith("0"):
            return "PISpot_SPS"

    if len(s) == 3:
        return "ALL"

    return ""


def _add_router_total_density(clean_df: pd.DataFrame, api_cfg: API_Config) -> pd.DataFrame:
    """
    在 router 端重算 Total defect count / Total glass count / Total density。

    計算語意：
      1. 後端 density_summary 是 per recipe_id + per adc_def_code。
      2. 先在 recipe 粒度上去除同 recipe 多 defect code 造成的 glass_cnt 重複。
      3. 再依 tab family 聚合 total。
      4. merge 回原本每一列，讓前端直接取用。
    """
    if clean_df is None or clean_df.empty:
        clean_df = clean_df.copy()
        clean_df["total_defect_cnt"] = 0
        clean_df["total_glass_cnt"] = 0
        clean_df["total_density"] = 0.0
        return clean_df

    df = clean_df.copy()

    base_keys = ["pi_hour", "line_id", "aoi", "model", "glass_type"]
    recipe_keys = base_keys + ["recipe_id"]

    need_cols = set(base_keys + ["recipe_id", "adc_def_code", "glass_cnt", "defect_cnt"])
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        print("miss")
        df["total_defect_cnt"] = 0
        df["total_glass_cnt"] = 0
        df["total_density"] = 0.0
        return df

    df["recipe_family"] = df["recipe_id"].apply(_recipe_family)
    # 給 merge 用：ALL 要先暫時歸到目前 row 所屬的圖表用途
    # 若 adc_def_code 是 SPS，歸 SPS；其他三碼預設先給 PISpot_SPS
    def _merge_family(row):
        fam = row.get("recipe_family", "")
        code = str(row.get("adc_def_code", "")).strip()

        if fam in ["UPI", "PISpot_SPS"]:
            return fam

        if fam == "ALL":
            if code in getattr(api_cfg, "uni_UPI_defect_codes", []):
                return "UPI"
            if code in getattr(api_cfg, "uni_SPOT_defect_codes", []):
                return "PISpot_SPS"
            if code in getattr(api_cfg, "uni_SPS_defect_codes", []):
                return "PISpot_SPS"
            return "PISpot_SPS"

        return fam
    df["merge_family"] = df.apply(_merge_family, axis=1)

    df["glass_cnt"] = pd.to_numeric(df["glass_cnt"], errors="coerce").fillna(0)
    df["defect_cnt"] = pd.to_numeric(df["defect_cnt"], errors="coerce").fillna(0)
    #print(df[["adc_def_code", "recipe_family", "glass_cnt", "defect_cnt"]])
    # per recipe 去重 glass_cnt，避免同一 recipe 下多 defect code 重複加總 glass_cnt
    recipe_base = (
        df.groupby(recipe_keys + ["recipe_family"], dropna=False)
          .agg(
              recipe_total_glass_cnt=("glass_cnt", "max"),
              recipe_total_defect_cnt=("defect_cnt", "sum"),
          )
          .reset_index()
    )

    # UPI
    upi_base = recipe_base[recipe_base["recipe_family"].isin(["UPI", "ALL"])].copy()
    upi_total = (
        upi_base.groupby(base_keys, dropna=False)
                .agg(
                    total_glass_cnt=("recipe_total_glass_cnt", "sum"),
                    total_defect_cnt=("recipe_total_defect_cnt", "sum"),
                )
                .reset_index()
    )
    upi_total["recipe_family"] = "UPI"
    #print( upi_total)
    # PISpot / SPS：目前你的 recipe 規則兩者共用 0xxx + 3碼
    pis_sps_base = recipe_base[recipe_base["recipe_family"].isin(["PISpot_SPS", "ALL"])].copy()
    pis_sps_total = (
        pis_sps_base.groupby(base_keys, dropna=False)
                    .agg(
                        total_glass_cnt=("recipe_total_glass_cnt", "sum"),
                        total_defect_cnt=("recipe_total_defect_cnt", "sum"),
                    )
                    .reset_index()
    )
    pis_sps_total["recipe_family"] = "PISpot_SPS"

    
    upi_total["merge_family"] = "UPI"
    pis_sps_total["merge_family"] = "PISpot_SPS"


    total_df = pd.concat([upi_total, pis_sps_total], ignore_index=True)
    
    if total_df.empty:
        df["total_defect_cnt"] = 0
        df["total_glass_cnt"] = 0
        df["total_density"] = 0.0
        return df.drop(columns=["recipe_family"], errors="ignore")

    total_df["total_density"] = (
        total_df["total_defect_cnt"] /
        total_df["total_glass_cnt"].replace(0, pd.NA)
    ).fillna(0).round(3)
    print( total_df[["total_defect_cnt", "total_glass_cnt", "total_density"]])
    df = df.merge(
        total_df,
        on=base_keys + ["merge_family"],
        how="left"
    )
    print(df[["total_defect_cnt", "total_glass_cnt", "total_density"]])
    df["total_defect_cnt"] = df["total_defect_cnt"].fillna(0)
    df["total_glass_cnt"] = df["total_glass_cnt"].fillna(0)
    df["total_density"] = df["total_density"].fillna(0.0)

    return df.drop(columns=["recipe_family", "merge_family"], errors="ignore")


In [9]:
from cim_density_job import Config as DensityJobConfig

db = MySQLConnet('piaoi_density')
#db.list_tables()
#df = db.get_table('rtms_aoi300_raw_202605')
tbn = 'density_code_summary_202605'
cim_job_cfg = DensityJobConfig()
#api_cfg = API_Config(cim_job_cfg)
#df = db.get_runs_between(tbn,  '2026-05-02', '2026-05-05', time_col="pi_hour")
rows = db.get_rows(tbn, {'line_id': 'CAPIC200', 'aoi':'aoi200', 'model': 'M315QAN01', 'recipe_id':'2287', 'pi_hour': '2026-05-09 07:00:00'})
rows

[{'line_id': 'CAPIC200',
  'aoi': 'aoi200',
  'model': 'M315QAN01',
  'glass_type': 'TFT',
  'pi_hour': datetime.datetime(2026, 5, 9, 7, 0),
  'recipe_id': '2287',
  'adc_def_code': 'Polymer',
  'recipe_total_glass_cnt': 7,
  'recipe_total_defect_cnt': 768,
  'recipe_total_density': 109.714,
  'defect_cnt': 5,
  'def_glass_cnt': 3,
  'glass_cnt': 7,
  'recipe_code_density': 0.714,
  'density': 0.714,
  'small_defect_count': 0,
  'middle_defect_count': 4,
  'large_defect_count': 1,
  'over_defect_count': 0,
  'glass': '3D6A3Z96A,3D6A3Z96B,3D6A3Z96C,3D6A3Z96D,3D6A3Z96E,3D6A3Z96F,3D6A3Z96G',
  'glass_size_detail': '{"3D6A3Z96A": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "3D6A3Z96B": {"S": 0, "M": 1, "L": 0, "O": 0, "T": 1}, "3D6A3Z96C": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "3D6A3Z96D": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "3D6A3Z96E": {"S": 0, "M": 1, "L": 1, "O": 0, "T": 2}, "3D6A3Z96F": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "3D6A3Z96G": {"S": 0, "M": 2, "L": 0, "O": 0, "T"

In [28]:
980/6

163.33333333333334

In [38]:
clean_df = _add_router_total_density(df, api_cfg)
clean_df

   total_defect_cnt  total_glass_cnt  total_density
0               980                6        163.333
1               980                6        163.333
   total_defect_cnt  total_glass_cnt  total_density
0               980                6        163.333
1               980                6        163.333
2               980                6        163.333
3               980                6        163.333
4               980                6        163.333
5               980                6        163.333
6               980                6        163.333
7               980                6        163.333


,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,defect_cnt,glass_cnt,density,...,shift,comment,action,Editor,modify_time,recipe_family_x,total_glass_cnt,total_defect_cnt,recipe_family_y,total_density
0,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,NPI_CF,108,6,18.000,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333
1,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,NPI_OIL,1,6,0.167,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333
2,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,Not_Found,18,6,3.000,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333
3,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,OP_Defect,48,6,8.000,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333
4,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,PI_Spot_NP,1,6,0.167,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333
5,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,Polymer,53,6,8.833,...,{},,,,2026-05-05 15:05:04,ALL,6,980,UPI,163.333
6,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,SSIU_Polymer,698,6,116.333,...,{},,,,2026-05-05 15:05:04,ALL,6,980,UPI,163.333
7,CAPIC200,aoi100,G215HVN01,CF,608,2026-05-05 10:00:00,others,53,6,8.833,...,{},,,,2026-05-05 15:05:04,ALL,6,980,PISpot_SPS,163.333


In [5]:
df[(df['aoi'] =='aoi100' )& (df['model'] == 'G215HVN01') ] # & (df['pi_hour'] == '2026-05-05 10:00:00')

,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,defect_cnt,glass_cnt,density,...,middle_defect_count,large_defect_count,over_defect_count,glass,glass_size_detail,shift,comment,action,Editor,modify_time


In [11]:
db = MySQLConnet('rtms_piaoi_other')
db.list_tables()
df = db.get_table('rtms_aoi300_raw_202605')

#df['defect_size'].unique()

2026-05-07 13:34:48,815 - INFO - [list_tables] 成功取得資料表名稱，共 4 張表。


In [12]:
for (line, pi_type, ) , rows in df.groupby(['line_id', 'pi_type']):
    print(line, pi_type, len(rows))

Null API 144406
Null BPI 60656
Null CELL-ITO 824


In [11]:
rows = df[df['sheet_id_chip_id'] =='ZG6A5N17C']
print(len(rows))
rows['test_time'].unique()
rows.iloc[0].to_dict()

191


{'id': 4964,
 'sheet_id_chip_id': 'ZG6A5N17C',
 'chip_id': 'ZG6A5N17C2',
 'test_time': Timestamp('2026-05-04 16:21:16'),
 'defect_size': 'S',
 'size_class': '6',
 'pox_x1': 310572,
 'pox_y1': 1035260,
 'adc_def_code': '',
 'retype_def_code': '',
 'image_file_name': 'CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.1.bt5.1777882876.jpg',
 'img_file_url_path': 'PIT/2605/04/CAAOI300/ZG6A5N17C/1621/',
 'pic_path': 'http://10.97.139.98:1454/CAAOI300/CA020601/ZG6A5N17C/PCS1/58703/CaptureImage/small/CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.1.bt5.1777882876.jpg',
 'recipe_id': 'M270QAN06-T-API-260225',
 'line_id': 'Null',
 'aoi': 'aoi300',
 'model': 'M270QAN06',
 'glass_type': 'TFT',
 'pi_time': NaT,
 'pi_type': 'API',
 'cst_id': 'CA020601',
 'defect_count': 190,
 'defect_id': '1',
 'source_file': 'CAAOI300.M270QAN06.M270QAN06-T-API-260225.CA020601.ZG6A5N17C.58703.defect',
 'source_mtime': Timestamp('2026-05-04 16:23:27'),
 'update_time': Timestamp('2026-05-05 0

In [16]:
df['recipe_id'].unique()

array(['M270DAN7F-T-API', 'M270QAN06-T-API-260225',
       'B160UAN4A-T-API-20260203', 'B180UAN01-T-API-260405',
       'M270QAN07-T-API_260410', 'M340QAR1B-T-API-060206',
       'M270TAN2Z-T-API-260416', 'M270HAN03-T-API', 'M270HAN1C-T-API',
       'B140UAN04-T-API', 'CELL-ITO', 'M215HAN01-T-API-060204',
       'B160UAN4A-T-BPI'], dtype=object)

In [ ]:
cols = ['aoi', 'test_time', 'glass_type', 'glass_type', 'defect_count'] #same gld
for (gld,  ) , rows in df.groupby(['sheet_id_chip_id']):
    print(gld, len(rows))
    print(rows[cols])


MQ6A5TK12 4
        aoi           test_time glass_type glass_type  defect_count
368  aoi300 2026-05-02 08:30:45        ITO        ITO             0
369  aoi300 2026-05-02 08:30:45        ITO        ITO             0
370  aoi300 2026-05-02 08:30:45        ITO        ITO             0
371  aoi300 2026-05-02 08:30:45        ITO        ITO             0
MQ6A5TK13 4
        aoi           test_time glass_type glass_type  defect_count
372  aoi300 2026-05-02 08:26:03        ITO        ITO             0
373  aoi300 2026-05-02 08:26:03        ITO        ITO             0
374  aoi300 2026-05-02 08:26:03        ITO        ITO             0
375  aoi300 2026-05-02 08:26:03        ITO        ITO             0
MQ6A5TK14 4
        aoi           test_time glass_type glass_type  defect_count
376  aoi300 2026-05-02 08:21:10        ITO        ITO             0
377  aoi300 2026-05-02 08:21:10        ITO        ITO             0
378  aoi300 2026-05-02 08:21:10        ITO        ITO             0
379  aoi300 

In [22]:
bill = [{'t':6242, 'm':3150},
{'t':31500, 'm':(700+2915+4580+1700+7020+901+900)},
{'t':21143, 'm':(3308+240+629)},
{'t':19860, 'm':0}, #'m':(-2860)
#{'t':3419, 'm':0}
]
total = 0
total_m = 0
for data in bill:
    t, m = data.values()
    t1 = t-m
    print(t1, m)
    total = total+ t1
    total_m = total_m + m

print(total, total_m, total-total_m)


3092 3150
12784 18716
16966 4177
19860 0
52702 26043 26659


In [6]:
db.get_rows('rtms_aoi300_raw_202604',{'defect_size': 'OK'})

[{'id': 17860074,
  'sheet_id_chip_id': 'MQ6A5PN71',
  'chip_id': 'MQ6A5PN710',
  'test_time': datetime.datetime(2026, 4, 24, 13, 7, 26),
  'defect_size': 'OK',
  'size_class': '0',
  'pox_x1': 0,
  'pox_y1': 0,
  'adc_def_code': '',
  'retype_def_code': '',
  'image_file_name': 'CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.GlassImage_modality5.bm.1777007246.jpg',
  'img_file_url_path': 'PIT/2604/24/CAAOI300/MQ6A5PN71/1307/Map/',
  'pic_path': 'http://10.97.139.98:1454/CAAOI300/CA002001/MQ6A5PN71/PCS1/47086/Map/CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.GlassImage_modality5.bm.1777007246.jpg',
  'recipe_id': 'CELL-ITO',
  'line_id': 'Null',
  'aoi': 'aoi300',
  'model': 'CELL-ITO',
  'glass_type': 'ITO',
  'pi_time': None,
  'pi_type': 'CELL-ITO',
  'cst_id': 'CA002001',
  'defect_count': 0,
  'defect_id': 'MACRO_1',
  'source_file': 'CAAOI300.CELL-ITO.CELL-ITO.CA002001.MQ6A5PN71.47086.defect',
  'source_mtime': datetime.datetime(2026, 4, 24, 13, 9, 27),
  'update_time': datetime.d

In [4]:
60000-22957-10600-10000-29733

-13290

In [17]:
db = MySQLConnet('piaoi_density')
db.list_tables()

2026-05-07 14:54:44,146 - INFO - [list_tables] 成功取得資料表名稱，共 10 張表。


['default_spec_table',
 'density_summary_202601',
 'density_summary_202602',
 'density_summary_202603',
 'density_summary_202604',
 'density_summary_202605',
 'fix_spec_table_202602',
 'fix_spec_table_202603',
 'fix_spec_table_202604',
 'fix_spec_table_202605']

In [22]:
rows = db.get_rows('density_summary_202605', {'line_id': 'CAPIC200','aoi': 'aoi200', 'model':'M270TAN01'})

pd.DataFrame(rows)

,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,defect_cnt,def_glass_cnt,small_defect_count,...,over_defect_count,glass_cnt,density,glass,glass_size_detail,shift,comment,action,Editor,modify_time
0,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,Polymer,0,0,0,...,0,5,0.000,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
1,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,SSIU_Polymer,2,2,2,...,0,5,0.400,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
2,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,PI_Spot_NP,0,0,0,...,0,5,0.000,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
3,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,PIS With Particle,0,0,0,...,0,5,0.000,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
4,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,SPS,0,0,0,...,0,5,0.000,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
5,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,NPI_TFT,4,3,3,...,0,5,0.800,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
6,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 05:00:00,Peeling,1,1,0,...,0,5,0.200,"T26A4YD4A,T26A4YD4B,T26A4YD4C,T26A4YD4E,T26A4YD4F","{""T26A4YD4A"": {""S"": 0, ""M"": 1, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
7,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 16:00:00,Polymer,1,1,0,...,0,7,0.143,"T26A3YS2B,T26A3YS2C,T26A3YS2D,T26A3YS2E,T26A3Y...","{""T26A3YS2B"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
8,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 16:00:00,SSIU_Polymer,2,2,2,...,0,7,0.286,"T26A3YS2B,T26A3YS2C,T26A3YS2D,T26A3YS2E,T26A3Y...","{""T26A3YS2B"": {""S"": 1, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20
9,CAPIC200,aoi200,M270TAN01,TFT,0296,2026-05-02 16:00:00,PI_Spot_NP,0,0,0,...,0,7,0.000,"T26A3YS2B,T26A3YS2C,T26A3YS2D,T26A3YS2E,T26A3Y...","{""T26A3YS2B"": {""S"": 0, ""M"": 0, ""L"": 0, ""O"": 0,...",{},,,,2026-05-07 14:59:20


In [ ]:
#data = {}
for tbn in ['density_summary_202605']:
    data[tbn] = db.get_table(tbn)

In [ ]:
import pandas as pd

()
def build_expand_spec_df(
    model_list,
    line_id_list=None,
    glass_type_list=None,
    size_group_list=None,
    default_ooc=150,
    default_oos=200,
    editor="預設",
    modify_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
):
    #if line_id_list is None:
    #    line_id_list = ["ALL"]

    if glass_type_list is None:
        glass_type_list = ["TFT", "CF"]

    if size_group_list is None:
        size_group_list = ["S", "MS", "LMS", "O", "OL", "OLM", "OLMS"]

    def clean_list(values):
        out = []
        for v in values or []:
            if v is None:
                continue
            s = str(v).strip()
            if not s:
                continue
            if s.lower() in {"nan", "none", "null", "<na>", "nat"}:
                continue
            out.append(s)
        return sorted(set(out))

    models = clean_list(model_list)
    #lines = clean_list(line_id_list)
    glass_types = clean_list(glass_type_list)
    size_groups = clean_list(size_group_list)

    rows = []

    #for line_id in lines:
    for model in models:
        for glass_type in glass_types:
            for size_group in size_groups:
                rows.append({
                    "model": model,
                    "glass_type": glass_type,
                    "defect_size": size_group,
                    "OOC": default_ooc,
                    "OOS": default_oos,
                    "Editor": editor,
                    "modify_time": modify_time ,
                    "drop" : 'F'
                })

    return pd.DataFrame(rows)

spec_df = build_expand_spec_df([v for v in models if 'test' not in v])
spec_df

,model,glass_type,defect_size,OOC,OOS,Editor,modify_time,drop
0,B070AAN01,CF,LMS,150,200,預設,2026-04-28 13:57:13,F
1,B070AAN01,CF,MS,150,200,預設,2026-04-28 13:57:13,F
2,B070AAN01,CF,O,150,200,預設,2026-04-28 13:57:13,F
3,B070AAN01,CF,OL,150,200,預設,2026-04-28 13:57:13,F
4,B070AAN01,CF,OLM,150,200,預設,2026-04-28 13:57:13,F
...,...,...,...,...,...,...,...,...
2585,Z335XAN01_TEST,TFT,O,150,200,預設,2026-04-28 13:57:13,F
2586,Z335XAN01_TEST,TFT,OL,150,200,預設,2026-04-28 13:57:13,F
2587,Z335XAN01_TEST,TFT,OLM,150,200,預設,2026-04-28 13:57:13,F
2588,Z335XAN01_TEST,TFT,OLMS,150,200,預設,2026-04-28 13:57:13,F


In [23]:
spec_df.columns

Index(['model', 'glass_type', 'defect_size', 'OOC', 'OOS', 'Editor',
       'modify_time', 'drop'],
      dtype='object')

In [3]:
db = MySQLConnet('piaoi_bpi_density')
#db.list_tables()
#db.save_table('default_spec_table', spec_df)
db.get_rows('default_spec_table', {'model': 'test_0429'})

[{'model': 'test_0429',
  'glass_type': 'TFT',
  'defect_size': 'MS',
  'OOC': 111,
  'OOS': 222,
  'Editor': '預設',
  'modify_time': '2026-04-29 10:33:03',
  'drop': 'F'}]

In [5]:
df = db.get_runs_between('bpi_api_summary_202604', '2026-04-26', '2026-04-28', 'scan_hour')
df.iloc[-1,:].to_dict()

{'id': 3,
 'aoi': 'aoi200',
 'model': 'B160UAN05',
 'scan_hour': Timestamp('2026-04-26 11:00:00'),
 'cassette_id': 'AA1659',
 'glass_side': 'TFT',
 'recipe_id': '4258',
 'pi_type': 'BPI',
 'run_day': datetime.date(2026, 4, 26),
 'glass_count': 1,
 'total_defect_count': 4851,
 'small_defect_count': 473,
 'middle_defect_count': 1563,
 'large_defect_count': 1359,
 'over_defect_count': 0,
 'density': 4851.0,
 'glass_list': 'VN6A3VE6A',
 'glass_size_detail': '{"VN6A3VE6A": {"S": 473, "M": 1563, "L": 1359, "O": 0, "T": 3395, "line_id": "CAPIC200", "test_time": "2026-04-26 12:09:45"}}',
 'source_db': 'cim_piaoi',
 'source_table': 'cim_pi_glass_202604',
 'comment': '',
 'action': '',
 'editor': '',
 'modify_time': None,
 'update_time': Timestamp('2026-04-28 11:08:34')}

In [10]:
db = MySQLConnet('cim_piaoi')
df = db.get_runs_between('cim_pi_glass_202604', '2026-04-26', '2026-04-27', 'pi_hour')
df.columns

Index(['sheet_id_chip_id', 'test_time', 'model_no', 'op_id', 'abbr_cat',
       'recipe_id', 'cassette_id', 'aoi', 'total_defect_qty',
       'defect_size_o_qty', 'defect_size_l_qty', 'defect_size_m_qty',
       'defect_size_s_qty', 'line_id', 'pi_time', 'pi_hour', 'pi_type'],
      dtype='object')

In [8]:
rdf = df[(df['cassette_id'] == 'AA1659') & (df['model_no'] == 'B160UAN05') ] #(df['model'] == 'B160UAN05') & (df['scan_hour'] == "2026-04-26 11:00:00") 
rdf

,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
89,VN6A3VE6A,2026-04-26 12:09:45,B160UAN05,PCS1,TFT,4258,AA1659,CAAOI202,4851,0,1359,1563,473,CAPIC200,2026-04-26 18:21:21,2026-04-26 17:00:00,BPI


In [55]:
730*0.65-(120+22+13)
#100-3-(4*1.5)

319.5

In [ ]:
rdf = df[(df['cassette_id'] == "AA1659") ] #(df['model'] == 'B160UAN05') & (df['scan_hour'] == "2026-04-26 11:00:00") 

for (recipe, glass_type, code, ), rows in rdf.groupby(['cassette_id','recipe_id',  'glass_side']):
    print(recipe, glass_type,  code, len(rows))
    print(rows[['glass_count', 'total_defect_count',
       'small_defect_count', 'middle_defect_count', 'large_defect_count',
       'over_defect_count', 'density']])

KeyError: 'glass_side'

In [41]:
473+1563+1359

3395

In [13]:
db = MySQLConnet('piaoi_capa')
for tbn in [v for v in db.list_tables() if 'aoi300' not in v]:
    db.drop_table(tbn)

2026-04-22 15:00:31,584 - INFO - [list_tables] 成功取得資料表名稱，共 9 張表。
2026-04-22 15:00:32,545 - INFO - [drop_table] 資料表 'aoi100_202604_capa_glass_table' 已刪除.
2026-04-22 15:00:33,565 - INFO - [drop_table] 資料表 'aoi100_202604_capa_hourly_rawdata' 已刪除.
2026-04-22 15:00:34,252 - INFO - [drop_table] 資料表 'aoi100_202604_capa_summary' 已刪除.
2026-04-22 15:00:34,753 - INFO - [drop_table] 資料表 'aoi200_202604_capa_glass_table' 已刪除.
2026-04-22 15:00:35,358 - INFO - [drop_table] 資料表 'aoi200_202604_capa_hourly_rawdata' 已刪除.
2026-04-22 15:00:36,190 - INFO - [drop_table] 資料表 'aoi200_202604_capa_summary' 已刪除.


In [6]:

#tbn =  'cim_defect_202604_aoi200_pi000'
tbn =  'cim_pi_glass_202604'
df = cim_db.get_table(tbn)
print(len(df))
df.iloc[0,:].to_dict()#.head()

8799


{'sheet_id_chip_id': 'AJ5R130WLW',
 'test_time': Timestamp('2026-04-01 03:29:30'),
 'model_no': 'M195RTN01_RG',
 'op_id': 'PI AOI',
 'abbr_cat': 'CF',
 'recipe_id': '803',
 'cassette_id': 'CA0025',
 'aoi': 'CAPIT203',
 'total_defect_qty': 221,
 'defect_size_o_qty': 0,
 'defect_size_l_qty': 1,
 'defect_size_m_qty': 13,
 'defect_size_s_qty': 207,
 'line_id': 'CAPIC700',
 'pi_time': Timestamp('2026-04-01 01:28:10'),
 'pi_hour': Timestamp('2026-04-01 00:00:00'),
 'pi_type': 'API'}

In [5]:
cim_rdf = cim_db.get_runs_delta_days('cim_defect_202604_aoi200_capic200', 3, 'test_time')
cim_rdf.iloc[0,:].to_dict()

{'sheet_id_chip_id': 'VH6A3T79A',
 'chip_id': 'C',
 'test_time': Timestamp('2026-04-23 11:52:15'),
 'defect_size': 'S',
 'pox_x1': 295477,
 'pox_y1': 546681,
 'image_file_path': 'Image/CA0094/VH6A3T79A/',
 'image_file_name': 'RV1_295477_546681_0.jpg',
 'img_file_url_path': 'PIT/2604/23/CAAOI202/VH6A3T79A/1152/',
 'retype_def_code': 'nan',
 'adc_def_code': 'PI_Spot_NP',
 'pi_time': Timestamp('2026-04-23 09:59:06'),
 'pi_hour': Timestamp('2026-04-23 09:00:00')}

In [8]:
other_db = MySQLConnet('rtms_piaoi_other')
other_db.list_tables()

2026-04-23 13:41:26,009 - INFO - [list_tables] 成功取得資料表名稱，共 2 張表。


['rtms_aoi300_glass_202604', 'rtms_aoi300_raw_202604']

In [9]:
raws = other_db.get_table('rtms_aoi300_raw_202604')
raws['pi_type'].unique()

for tbn in other_db.list_tables():
    print(tbn)
    rows = other_db.get_runs_delta_days(tbn, 3, "test_time")
    print(rows.iloc[0,:].to_dict())

2026-04-23 13:41:38,662 - INFO - [list_tables] 成功取得資料表名稱，共 2 張表。


rtms_aoi300_glass_202604
{'id': 30607, 'sheet_id_chip_id': 'VH6A3KH2Q', 'test_time': Timestamp('2026-04-23 13:24:19'), 'recipe_id': 'M270TAN2Z-T-API-260416', 'line_id': 'Null', 'aoi': 'aoi300', 'model': 'M270TAN2Z', 'glass_type': 'TFT', 'pi_time': None, 'pi_type': 'API', 'defect_count': 101, 'small_defect_count': 101, 'middle_defect_count': 0, 'large_defect_count': 0, 'over_defect_count': 0, 'run_day': Timestamp('2026-04-23 00:00:00'), 'update_time': Timestamp('2026-04-23 13:34:42')}
rtms_aoi300_raw_202604
{'id': 15020888, 'sheet_id_chip_id': 'VH6A3KH2Q', 'chip_id': 'VH6A3KH2Q11', 'test_time': Timestamp('2026-04-23 13:24:19'), 'defect_size': 'S', 'size_class': '44', 'pox_x1': 31973, 'pox_y1': 455827, 'adc_def_code': '', 'retype_def_code': '', 'image_file_name': 'CAAOI300.M270TAN2Z.M270TAN2Z-T-API-260416.CA0035.VH6A3KH2Q.1.bt5.1776921859.jpg', 'img_file_url_path': 'PIT/2604/23/CAAOI300/VH6A3KH2Q/1324/', 'pic_path': 'http://10.97.139.98:1454/CAAOI300/CA0035/VH6A3KH2Q/PCS1/48087/CaptureIm